# Phase 4 — Evaluation + Failure Analysis
Compares the Phase 2 popularity baseline against the Phase 3 final pipeline (ALS+Item-CF candidates -> XGBoost ranking) on the **held-out TEST set**. All artifacts (ALS, Item-CF, XGBoost, and the train-only feature tables) are reused exactly as trained in Phase 3 -- nothing is retrained here, matching Phase 5's "no live model calls" serving design.

## Cell 1: Load everything
Test set, train (for the baseline's exclusion logic), and all Phase 2/3 artifacts.

In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import json
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

import sys
sys.path.insert(0, "..")
from src.baseline import compute_popularity_score, compute_user_top_category, build_category_topk, recommend
from src.recommender import generate_candidates
from src.ranker import build_training_set, score_candidates, top_k, FEATURE_COLS
from src.evaluate import recall_at_k, ndcg_at_k, coverage_at_k

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
test = pd.read_parquet(PROCESSED_DIR / "test.parquet")

with open(PROCESSED_DIR / "split_dates.json") as f:
    split_dates = json.load(f)
T1, T2 = split_dates["T1_ms"], split_dates["T2_ms"]

item_category = pd.read_parquet(PROCESSED_DIR / "item_category_train.parquet").set_index("itemid")["categoryid"]
user_features = pd.read_parquet(PROCESSED_DIR / "user_features.parquet")
item_features = pd.read_parquet(PROCESSED_DIR / "item_features.parquet")
user_item_features = pd.read_parquet(PROCESSED_DIR / "user_item_features.parquet")

with open(PROCESSED_DIR / "als_model.pkl", "rb") as f:
    als_model = pickle.load(f)
with open(PROCESSED_DIR / "itemcf_matrix.pkl", "rb") as f:
    itemcf_model = pickle.load(f)
with open(PROCESSED_DIR / "id_mappings.pkl", "rb") as f:
    id_mappings = pickle.load(f)
with open(PROCESSED_DIR / "xgboost_ranker.pkl", "rb") as f:
    xgb_model = pickle.load(f)

user_to_idx, item_to_idx = id_mappings["user_to_idx"], id_mappings["item_to_idx"]
idx_to_user, idx_to_item = id_mappings["idx_to_user"], id_mappings["idx_to_item"]
best_alpha = id_mappings["best_alpha"]

# Rebuild the train user-item CSR matrix (not itself persisted; cheap to reconstruct
# from the already-cached user_item_features + the saved id mappings).
ui_weight = (
    user_item_features["user_item_views"] * 1
    + user_item_features["user_item_carts"] * 2
    + user_item_features["user_item_purchases"] * 4
).astype(np.float32)
row = user_item_features["visitorid"].map(user_to_idx).values
col = user_item_features["itemid"].map(item_to_idx).values
user_item_csr = csr_matrix((ui_weight.values, (row, col)), shape=(len(user_to_idx), len(item_to_idx)))

print("train:", train.shape, " test:", test.shape)
print("T1:", pd.to_datetime(T1, unit="ms"), " T2:", pd.to_datetime(T2, unit="ms"))
print("\nartifacts loaded: als_model, itemcf_model, xgb_model, id_mappings (best_alpha =", best_alpha, ")")
print("user_features:", user_features.shape, " item_features:", item_features.shape, " user_item_features:", user_item_features.shape)
print("user_item_csr:", user_item_csr.shape, " nnz:", user_item_csr.nnz)

train: (2204512, 6)  test: (275565, 6)
T1: 2015-08-18 04:23:20.445000  T2: 2015-09-02 17:49:14.032000

artifacts loaded: als_model, itemcf_model, xgb_model, id_mappings (best_alpha = 0.7 )
user_features: (1123765, 8)  item_features: (212915, 9)  user_item_features: (1713171, 7)
user_item_csr: (1123765, 212915)  nnz: 1713171


## Cell 2: Popularity baseline on test
Same train-only popularity model as Phase 2, applied to test users this time (never seen during training or tuning).

In [2]:
K = 10

popularity_score = compute_popularity_score(train)
global_topk = popularity_score.index.tolist()
user_top_category_baseline = compute_user_top_category(train, item_category)
category_topk = build_category_topk(popularity_score, item_category)
purchased_in_train = train[train["event"] == "transaction"].groupby("visitorid")["itemid"].apply(set).to_dict()

test_users = test["visitorid"].unique()
print(f"test users: {len(test_users):,}")

t0 = time.time()
recs_rows = []
for uid in test_users:
    recs = recommend(uid, K, global_topk, category_topk, user_top_category_baseline, purchased_in_train)
    for rank, itemid in enumerate(recs, start=1):
        recs_rows.append((uid, rank, itemid))
baseline_recs_test = pd.DataFrame(recs_rows, columns=["visitorid", "rank", "itemid"])
print(f"generated in {time.time()-t0:.1f}s")

baseline_recs_test.to_parquet(PROCESSED_DIR / "baseline_recs_test.parquet", index=False)
print(f"saved baseline_recs_test.parquet: {baseline_recs_test.shape}")
print("users covered:", baseline_recs_test["visitorid"].nunique(), "/", len(test_users))
print("mean recs per user:", baseline_recs_test.groupby("visitorid").size().mean())

test users: 158,020


generated in 5.3s


saved baseline_recs_test.parquet: (1580200, 3)
users covered: 158020 / 158020
mean recs per user: 10.0


## Cell 3: Final pipeline (ALS+Item-CF candidates -> XGBoost) on test
Candidates + ranking only exist for warm-in-test users (present in the train ALS/Item-CF fit) — same scoping constraint as Phase 3's val-side generation. Cold-start test users get no personalized output here; Cell 4 blends in the popularity fallback for them, matching the real serving design.

In [3]:
test_users_set = set(test["visitorid"].unique())
warm_test_mask = np.isin(idx_to_user, np.array(list(test_users_set)))
warm_test_user_idx = np.nonzero(warm_test_mask)[0]
print(f"train users also present in test (warm): {len(warm_test_user_idx):,} / {len(idx_to_user):,} train users")
print(f"  = {len(warm_test_user_idx) / len(test_users_set):.1%} of {len(test_users_set):,} test users")

train_purchases = train[train["event"] == "transaction"][["visitorid", "itemid"]].drop_duplicates()
train_purchases["user_idx"] = train_purchases["visitorid"].map(user_to_idx)
train_purchases["item_idx"] = train_purchases["itemid"].map(item_to_idx)
purchased_item_idx = train_purchases.groupby("user_idx")["item_idx"].apply(set).to_dict()

t0 = time.time()
candidate_pool_test = generate_candidates(
    als_model, itemcf_model, user_item_csr, warm_test_user_idx, purchased_item_idx, candidate_n=50,
)
print(f"\ncandidate retrieval done in {time.time()-t0:.1f}s, pool shape: {candidate_pool_test.shape}")

candidate_pool_test["itemid"] = idx_to_item[candidate_pool_test["item_idx"].values]
candidate_pool_test["visitorid"] = idx_to_user[candidate_pool_test["user_idx"].values]
candidate_pool_test["candidate_score"] = (
    best_alpha * candidate_pool_test["als_score"] + (1 - best_alpha) * candidate_pool_test["itemcf_score"]
)

final_candidates_test = (
    candidate_pool_test.sort_values(["visitorid", "candidate_score"], ascending=[True, False])
    .groupby("visitorid").head(50).copy()
)
print(f"final candidates per user: mean={final_candidates_test.groupby('visitorid').size().mean():.1f}")

scored_test = build_training_set(final_candidates_test, user_features, item_features, user_item_features)
scored_test["xgb_score"] = score_candidates(xgb_model, scored_test)

personalized_recs_test = top_k(scored_test, "xgb_score", k=K)[["visitorid", "itemid", "xgb_score", "rank"]]
personalized_recs_test.to_parquet(PROCESSED_DIR / "personalized_recs_test.parquet", index=False)
print(f"\npersonalized_recs_test shape: {personalized_recs_test.shape} -- saved personalized_recs_test.parquet")
print("users covered:", personalized_recs_test["visitorid"].nunique())
print("mean recs per user:", personalized_recs_test.groupby("visitorid").size().mean())

train users also present in test (warm): 11,587 / 1,123,765 train users
  = 7.3% of 158,020 test users



candidate retrieval done in 9.5s, pool shape: (852514, 5)


final candidates per user: mean=50.0



personalized_recs_test shape: (115870, 4) -- saved personalized_recs_test.parquet
users covered: 11587
mean recs per user: 10.0


## Cell 4: Blend with the popularity fallback
This is exactly Phase 5's serving logic: personalized top-10 for warm users, popularity top-10 for everyone else. Save `final_recs_test.parquet` as the deployed system's actual output.

In [4]:
warm_visitorids = set(personalized_recs_test["visitorid"].unique())

cold_fallback = baseline_recs_test[~baseline_recs_test["visitorid"].isin(warm_visitorids)][["visitorid", "itemid", "rank"]]
personalized_part = personalized_recs_test[["visitorid", "itemid", "rank"]]
final_recs_test = pd.concat([personalized_part, cold_fallback], ignore_index=True)
final_recs_test.to_parquet(PROCESSED_DIR / "final_recs_test.parquet", index=False)

print(f"final_recs_test: {final_recs_test.shape}, users covered: {final_recs_test['visitorid'].nunique():,} / {len(test_users):,}")
print(f"  personalized (warm, XGBoost-ranked): {len(warm_visitorids):,} users ({len(warm_visitorids)/len(test_users):.1%})")
print(f"  fallback (cold, popularity):         {final_recs_test['visitorid'].nunique() - len(warm_visitorids):,} users")

final_recs_test: (1580200, 3), users covered: 158,020 / 158,020
  personalized (warm, XGBoost-ranked): 11,587 users (7.3%)
  fallback (cold, popularity):         146,433 users


## Cell 5: Evaluate on test
Two comparisons, both against the same test-set ground truth:
1. **Main (roadmap-required):** popularity baseline vs. the deployed final pipeline (personalized + fallback blended), over all 158,020 test users.
2. **Supplementary:** restricted to the 11,587 warm users, baseline-for-those-users vs. personalized-for-those-users — isolates what personalization itself contributes, since (1) is diluted by the 92.7% of users where both systems produce identical (fallback) output.

In [5]:
def recs_dict(df):
    return df.sort_values(["visitorid", "rank"]).groupby("visitorid")["itemid"].apply(list).to_dict()

baseline_by_user = recs_dict(baseline_recs_test)
final_by_user = recs_dict(final_recs_test)

relevant_any = test.groupby("visitorid")["itemid"].apply(set).to_dict()
relevant_cart = test[test["event"] == "addtocart"].groupby("visitorid")["itemid"].apply(set).to_dict()
relevant_purchase = test[test["event"] == "transaction"].groupby("visitorid")["itemid"].apply(set).to_dict()
catalog_size = train["itemid"].nunique()

def eval_system(recs_by_user, relevant_any_, relevant_cart_, relevant_purchase_):
    recall, _ = recall_at_k(recs_by_user, relevant_any_)
    ndcg, _ = ndcg_at_k(recs_by_user, relevant_any_, K)
    coverage, _ = coverage_at_k(recs_by_user, catalog_size)
    atc_recall, _ = recall_at_k(recs_by_user, relevant_cart_)
    purch_recall, _ = recall_at_k(recs_by_user, relevant_purchase_)
    return {
        "recall_at_10": recall, "ndcg_at_10": ndcg, "coverage_at_10": coverage,
        "addtocart_recall_at_10": atc_recall, "purchase_recall_at_10": purch_recall,
    }

print("=" * 70)
print("MAIN COMPARISON -- all 158,020 test users (deployed system)")
print("=" * 70)
baseline_metrics_test = eval_system(baseline_by_user, relevant_any, relevant_cart, relevant_purchase)
final_metrics_test = eval_system(final_by_user, relevant_any, relevant_cart, relevant_purchase)

print(f"{'Metric':<26}{'Baseline':>12}{'Final Pipeline':>16}{'Delta %':>12}")
for k in baseline_metrics_test:
    b, f = baseline_metrics_test[k], final_metrics_test[k]
    delta_pct = (f - b) / b * 100 if b != 0 else float("nan")
    print(f"{k:<26}{b:>12.4f}{f:>16.4f}{delta_pct:>11.1f}%")

with open(PROCESSED_DIR / "baseline_metrics_test.json", "w") as fh:
    json.dump(baseline_metrics_test, fh, indent=2)
with open(PROCESSED_DIR / "final_metrics_test.json", "w") as fh:
    json.dump(final_metrics_test, fh, indent=2)

MAIN COMPARISON -- all 158,020 test users (deployed system)


Metric                        Baseline  Final Pipeline     Delta %
recall_at_10                    0.0100          0.0144       43.7%
ndcg_at_10                      0.0059          0.0103       75.8%
coverage_at_10                  0.0344          0.1505      337.9%
addtocart_recall_at_10          0.0098          0.0134       36.2%
purchase_recall_at_10           0.0182          0.0247       35.2%


In [6]:
print("=" * 70)
print(f"SUPPLEMENTARY -- {len(warm_visitorids):,} warm users only (isolates personalization)")
print("=" * 70)

baseline_warm_by_user = {u: r for u, r in baseline_by_user.items() if u in warm_visitorids}
personalized_by_user = recs_dict(personalized_recs_test)
relevant_any_warm = {u: r for u, r in relevant_any.items() if u in warm_visitorids}
relevant_cart_warm = {u: r for u, r in relevant_cart.items() if u in warm_visitorids}
relevant_purchase_warm = {u: r for u, r in relevant_purchase.items() if u in warm_visitorids}

baseline_metrics_warm = eval_system(baseline_warm_by_user, relevant_any_warm, relevant_cart_warm, relevant_purchase_warm)
personalized_metrics_warm = eval_system(personalized_by_user, relevant_any_warm, relevant_cart_warm, relevant_purchase_warm)

print(f"{'Metric':<26}{'Baseline':>12}{'Personalized':>16}{'Delta %':>12}")
for k in baseline_metrics_warm:
    b, p = baseline_metrics_warm[k], personalized_metrics_warm[k]
    delta_pct = (p - b) / b * 100 if b != 0 else float("nan")
    print(f"{k:<26}{b:>12.4f}{p:>16.4f}{delta_pct:>11.1f}%")

SUPPLEMENTARY -- 11,587 warm users only (isolates personalization)


Metric                        Baseline    Personalized     Delta %
recall_at_10                    0.0432          0.1028      138.0%
ndcg_at_10                      0.0273          0.0879      221.5%
coverage_at_10                  0.0344          0.1505      337.9%
addtocart_recall_at_10          0.0280          0.0650      132.3%
purchase_recall_at_10           0.0445          0.0946      112.8%


## Cell 6: Failure analysis
All 7 modes measured from actual data, not asserted.

In [7]:
print("=" * 70)
print("1. COLD-START USERS")
print("=" * 70)
train_users_set = set(train["visitorid"].unique())
cold_test_users = test_users_set - train_users_set
print(f"users with 0 train interactions: {len(cold_test_users):,} / {len(test_users_set):,} "
      f"({len(cold_test_users)/len(test_users_set):.1%}) -- fall back to category-level popularity")

print("\n" + "=" * 70)
print("2. SPARSE USERS -- Recall@10 (final pipeline) by train-activity bucket")
print("=" * 70)
train_event_count = train.groupby("visitorid").size()
test_users_arr = np.array(list(test_users_set))
train_counts_arr = train_event_count.reindex(test_users_arr, fill_value=0).values
buckets = np.select(
    [train_counts_arr == 0, train_counts_arr <= 2, train_counts_arr <= 5],
    ["0 (cold)", "1-2 (sparse)", "3-5"],
    default="6+",
)
bucket_series = pd.Series(buckets, index=test_users_arr)
print(bucket_series.value_counts().reindex(["0 (cold)", "1-2 (sparse)", "3-5", "6+"]))
print()
for b in ["0 (cold)", "1-2 (sparse)", "3-5", "6+"]:
    users_in_bucket = set(bucket_series[bucket_series == b].index)
    rel_b = {u: r for u, r in relevant_any.items() if u in users_in_bucket}
    recall_b, n_b = recall_at_k(final_by_user, rel_b)
    print(f"  {b:<15} n_users_with_relevant={n_b:>7,}  Recall@10={recall_b:.4f}")

print("\n" + "=" * 70)
print("3. POPULARITY BIAS IN ALS")
print("=" * 70)
top100_popular = set(global_topk[:100])
als_sourced = candidate_pool_test[candidate_pool_test["candidate_source"].isin(["als", "both"])]
row_overlap = als_sourced["itemid"].isin(top100_popular).mean()
unique_als_items = als_sourced["itemid"].unique()
item_overlap = pd.Series(unique_als_items).isin(top100_popular).mean()
print(f"ALS-sourced candidate rows also in train's top-100 popular items: {row_overlap:.1%}")
print(f"unique ALS-recommended items ({len(unique_als_items):,}) overlapping top-100 popular: {item_overlap:.1%}")

print("\n" + "=" * 70)
print("4. LONG-TAIL ITEMS")
print("=" * 70)
item_total_interactions = item_features.set_index("itemid")[
    ["item_total_views", "item_total_carts", "item_total_purchases"]
].sum(axis=1)
longtail_items_set = set(item_total_interactions[item_total_interactions < 5].index)
popular_items_set = set(item_total_interactions[item_total_interactions >= 5].index)

relevant_longtail = {u: (r & longtail_items_set) for u, r in relevant_any.items()}
relevant_longtail = {u: r for u, r in relevant_longtail.items() if r}
relevant_popular = {u: (r & popular_items_set) for u, r in relevant_any.items()}
relevant_popular = {u: r for u, r in relevant_popular.items() if r}

recall_longtail, n_lt = recall_at_k(final_by_user, relevant_longtail)
recall_popular, n_pop = recall_at_k(final_by_user, relevant_popular)
print(f"Recall@10 when the relevant item is long-tail (<5 train interactions): {recall_longtail:.4f}  (n_users={n_lt:,})")
print(f"Recall@10 when the relevant item is popular (>=5 train interactions):  {recall_popular:.4f}  (n_users={n_pop:,})")

print("\n" + "=" * 70)
print("5. REPEATED RECOMMENDATIONS")
print("=" * 70)
viewed_df = user_item_features.loc[user_item_features["user_item_views"] > 0, ["visitorid", "itemid"]].assign(was_viewed=True)
merged_viewed = personalized_recs_test.merge(viewed_df, on=["visitorid", "itemid"], how="left")
repeat_rate = merged_viewed["was_viewed"].notna().mean()
print(f"of the {len(personalized_recs_test):,} personalized top-10 recs, {repeat_rate:.1%} are items the user")
print("already viewed (but did not purchase) in train -- purchased items are excluded by construction,")
print("but viewed-not-purchased items are allowed back in (Phase 3 Cell 8's documented fix).")

1. COLD-START USERS


users with 0 train interactions: 146,433 / 158,020 (92.7%) -- fall back to category-level popularity

2. SPARSE USERS -- Recall@10 (final pipeline) by train-activity bucket


0 (cold)        146433
1-2 (sparse)      7449
3-5               2166
6+                1972
Name: count, dtype: int64



  0 (cold)        n_users_with_relevant=146,433  Recall@10=0.0074
  1-2 (sparse)    n_users_with_relevant=  7,449  Recall@10=0.0795
  3-5             n_users_with_relevant=  2,166  Recall@10=0.1193
  6+              n_users_with_relevant=  1,972  Recall@10=0.1726

3. POPULARITY BIAS IN ALS


ALS-sourced candidate rows also in train's top-100 popular items: 27.3%
unique ALS-recommended items (3,925) overlapping top-100 popular: 2.4%

4. LONG-TAIL ITEMS


Recall@10 when the relevant item is long-tail (<5 train interactions): 0.0041  (n_users=27,012)
Recall@10 when the relevant item is popular (>=5 train interactions):  0.0176  (n_users=125,535)

5. REPEATED RECOMMENDATIONS


of the 115,870 personalized top-10 recs, 21.5% are items the user
already viewed (but did not purchase) in train -- purchased items are excluded by construction,
but viewed-not-purchased items are allowed back in (Phase 3 Cell 8's documented fix).


### 6. Temporal staleness
Unlike every other feature in this project, this one deliberately uses the **full** `item_properties` history (not filtered to timestamp < T1) — the question here is "is this item still available *right now*", which is exactly the real-time inventory check a serving system would run at request time, not a train-only feature.

In [8]:
avail_parts = []
for fname in ["item_properties_part1.csv", "item_properties_part2.csv"]:
    d = pd.read_csv(RAW_DIR / fname, dtype={"itemid": "int32", "property": "category", "value": "object"})
    d = d[d["property"] == "available"]
    avail_parts.append(d[["itemid", "timestamp", "value"]])
avail_raw = pd.concat(avail_parts, ignore_index=True)
latest_availability = avail_raw.sort_values("timestamp").drop_duplicates("itemid", keep="last").set_index("itemid")["value"]

print("latest 'available' value distribution across the full catalog:\n", latest_availability.value_counts())

recommended_items_final = set(final_recs_test["itemid"].unique())
avail_for_recommended = latest_availability.reindex(list(recommended_items_final))
unavailable_frac = (avail_for_recommended == "0").mean()
no_record_frac = avail_for_recommended.isnull().mean()

print(f"\nof {len(recommended_items_final):,} unique items recommended by the final system:")
print(f"  {unavailable_frac:.1%} show 'available'=0 in their most recent property record (would be filtered at serving time)")
print(f"  {no_record_frac:.1%} have no availability record at all")

latest 'available' value distribution across the full catalog:
 value
0    363043
1     54010
Name: count, dtype: int64

of 32,050 unique items recommended by the final system:
  56.2% show 'available'=0 in their most recent property record (would be filtered at serving time)
  6.7% have no availability record at all


### 7. Metric-business gap
Not a measurement — a limitation to state explicitly. Recall@10, NDCG@10, and the two behavioral-proxy recalls above are all computed on **historical, already-happened** holdout data (test). They quantify whether the model would have surfaced items the user *actually* went on to view/cart/purchase, in hindsight. None of that is the same claim as "this model would increase conversion rate or revenue if deployed" — that requires an online A/B test (treatment vs. control, live traffic, statistical significance), which this project does not run and does not claim. The relative improvements in Cell 5 (e.g. Recall@10 +43.7% blended, +138.0% for warm users) are offline ranking-quality deltas, not projected business impact.